The goal is to find similar functions that RDD has but with SQL

- `func.explode()` similar to flatmap; explodes columns into rows
- `func.split()`
- `func.lower
- Passing columns as parameters:
    - `func.split(inputDF.value, "\\W+")
    - filter(wordsDF.word != "")
    - Can also do func.col("columnName") to refer to a column

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

In [2]:
spark = SparkSession.builder.appName("WordCount").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/02 18:43:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/02 18:43:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/02 18:43:58 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/05/02 18:43:58 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/05/02 18:43:58 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/05/02 18:43:58 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/05/02 18:43:58 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.


In [3]:
inputDF = spark.read.text("/opt/spark/data/Book")

In [4]:
inputDF.show()

+--------------------+
|               value|
+--------------------+
|Self-Employment: ...|
|Achieving Financi...|
|       By Frank Kane|
|                    |
|                    |
|                    |
|Copyright � 2015 ...|
|All rights reserv...|
|                    |
|                    |
|            CONTENTS|
|          Disclaimer|
|             Preface|
|Part I: Making th...|
|  Overcoming Inertia|
|     Fear of Failure|
|Career Indoctrina...|
|The Carrot on a S...|
|      Ego Protection|
|Your Employer as ...|
+--------------------+
only showing top 20 rows



In [5]:
# Split using regular expression that extracts words

words = inputDF.select(func.explode(func.split(inputDF.value, "\\W+")).alias("word"))

In [6]:
words.show()

+----------+
|      word|
+----------+
|      Self|
|Employment|
|  Building|
|        an|
|  Internet|
|  Business|
|        of|
|       One|
| Achieving|
| Financial|
|       and|
|  Personal|
|   Freedom|
|   through|
|         a|
| Lifestyle|
|Technology|
|  Business|
|        By|
|     Frank|
+----------+
only showing top 20 rows



In [8]:
wordsWithoutEmptyString = words.filter(words.word != "")

In [9]:
#normalize everything to lowercase
lowercaseWords = wordsWithoutEmptyString.select(func.lower(wordsWithoutEmptyString.word).alias("word"))

In [10]:
# count the ocurrences of each word
wordCounts = lowercaseWords.groupBy("word").count()

In [13]:
wordCounts.show(10)

+-------------+-----+
|         word|count|
+-------------+-----+
|       online|   50|
|          few|   40|
|         some|  121|
|  requirement|    1|
|         hope|    5|
|        still|   65|
|        those|   68|
|      barrier|    2|
|indoctrinated|    1|
|       harder|    3|
+-------------+-----+
only showing top 10 rows



In [14]:
#sort by counts
wordCountsSorted = wordCounts.sort("count")

In [16]:
wordCountsSorted.show()

+-------------+-----+
|         word|count|
+-------------+-----+
|          125|    1|
| manipulation|    1|
|       graphs|    1|
|indoctrinated|    1|
|       column|    1|
|    traveling|    1|
|     slightly|    1|
| inflammatory|    1|
|   variations|    1|
|       spared|    1|
|          800|    1|
|    indicator|    1|
|        hires|    1|
|           07|    1|
|   surrounded|    1|
|     retailer|    1|
|          fax|    1|
|   afterwards|    1|
|        boost|    1|
|    directors|    1|
+-------------+-----+
only showing top 20 rows

